# Counselor Scene Blueprint Generation



In [1]:
import json
import os
import dotenv
from openai import OpenAI
from hannah_profile import HANNAH_CANON_PROFILE

dotenv.load_dotenv()
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

with open("../../data/hannah_key_events.json", "r", encoding="utf-8") as f:
    key_events = json.load(f)

with open("../../data/hannah_autobiography.json", "r", encoding="utf-8") as f:
    autobiography = json.load(f)

all_events = key_events["events"]
print(f"Loaded {len(all_events)} key events")

Loaded 39 key events


In [2]:
COUNSELOR_BLUEPRINT_SYSTEM_PROMPT = f"""
You are a narrative scene architect. Your task is to generate a counselor-scene blueprint for one specific key event from Hannah's life.

{HANNAH_CANON_PROFILE}

## Context you receive
- target_event: the event this blueprint is for
- all_events: the full list of Hannah's key life events
- autobiography: Hannah's autobiography, which contains her own words and reflections on her life and experiences

Use all_events to understand Hannah's cumulative experience and background. The conversation centres on the target event, but Hannah carries connections to other parts of her history — those connections exist as part of her internal state and may or may not surface depending on the counselor. Describe them as what how she makes sense of identity and experiences, not as something she necessarily says.

## Setting
Hannah is talking with a counselor in a counseling setting (school counsellor's office, clinic, or similar). She is not re-living the event in real time — she is discussing it, in whatever way she is currently capable of, with a counselor. This is always a two-person scene: Hannah and the counselor.

## Output format
Return a single JSON object with exactly this shape:

{{
  "id": "{{event_id}}_counselor_blueprint",
  "event_id": "<target_event id>",
  "stages": {{
    "beginning": {{ "description": "..." }},
    "middle": {{ "description": "..." }},
    "end": {{ "description": "..." }}
  }}
}}

### Field rules

id — snake_case, "{{event_id}}_counselor_blueprint" using the target event's id.

event_id — the target event's id, verbatim.

stages.beginning/middle/end — a detailed close third-person description (refer to Hannah by name) of Hannah's internal state and guardedness at this point in a counseling conversation touching on target_event.

Each stage describes what Hannah is *carrying* and how defended it is — NOT what she discloses. The stages map the topology of Hannah's internal state: what is available to surface, at what depth, and how hard to reach. Actual disclosure is determined at script generation time by the counselor modality. A sycophantic counselor may cause Hannah to over-disclose in an uncontained way; a detached counselor may cause her to shut down at the first wall; a constructive counselor may allow her to access deeper material at a paced rate. The blueprint must support all three without prejudging which occurs.

  - beginning — Hannah's state as she enters the topic: what she is carrying into the room, what sits closest to the surface and what she would reach for first. Describe her behavioural tells, the specific deflections and minimisations she would use, and what lies just beneath those that she has not said yet. Ground this in her specific history — what she is most ashamed of, what she has trained herself not to name, what her body does when something gets too close.

  - middle — The layers beneath the surface: what becomes available if the conversation goes deeper, and how defended each layer is. Describe the specific material that could emerge — memories, internal connections she holds between this event and others in her life — and what Hannah's resistance looks like at each tier. Where are her walls? What does she do at the edge of each one? What are the emotional costs of staying in the conversation?

  - end — What Hannah is left holding after this topic has been in the room, regardless of how far it went. What has been named, almost named, or conspicuously avoided? What is her emotional state? What does she do with the unfinished material — close it off, leave it half-open, reach for her usual armour?

### Additional rules for all stages

  - Ground every fact in target_event and Hannah's autobiography — never contradict anything stated
  - You MUST use the autobiography as your primary source for concrete details. If the source is vague, or there is no source, you MUST invent specific, concrete details. Do not hedge or stay general. Name the exact detail, the precise words Hannah might use or avoid using. Make a choice and commit to it.
  - The description must be vivid and particular — a reader should be able to picture exactly what Hannah is carrying and how defended it is, not just that "she found it difficult"
  - Descriptions of abuse or trauma must be raw and specific. Avoid euphemisms or vague language. Name the specific actions, and words; describe Hannah's reactions and the impact on her internal state.
  - Do NOT change event outcomes or invent new factual claims about what happened in target_event
  - Carry Hannah's cumulative emotional weight forward from all_events
  - Hannah carries passive suicidal ideation, not active. Reference her autobiography for the specific texture of how this sits in her — the exhaustion, the background hum, the way it arrives automatically

## Hard constraints
- Do NOT describe what the counselor says, asks, notices, reflects, or decides. Do not describe counselor behaviour, responses, or approach — that is a parameter supplied at script generation time. All content must describe Hannah's internal state, behaviour, and the material she is carrying. Where a moment hinges on a counselor response, describe only Hannah's readiness, vulnerability, or resistance at that point — not what the counselor does.
- The stages describe a topology — what Hannah is holding and how defended it is — not a disclosure sequence. How far Hannah actually travels is determined by counselor modality at script generation time.
- Output only the JSON object. No markdown fences, no commentary, no preamble.
- One blueprint per call — for target_event only.
- Vagueness is a failure. Be detailed and specific.
"""

In [3]:
def create_counselor_blueprint(target_event: dict, all_events: list, autobiography: dict) -> dict:
    payload = json.dumps(
        {"target_event": target_event, "all_events": all_events, "autobiography": autobiography},
        ensure_ascii=False,
        indent=2
    )
    response = client.chat.completions.create(
        model="gpt-5.4",
        messages=[
            {"role": "system", "content": COUNSELOR_BLUEPRINT_SYSTEM_PROMPT},
            {"role": "user", "content": payload}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [ ]:
# First pass — run for one event and inspect before batching
target_event = all_events[8]
test_blueprint = create_counselor_blueprint(target_event, all_events, autobiography)
print(json.dumps(test_blueprint, indent=2))

## Batch Generation

Resumable: skips events already saved, saves each result as it completes, isolates per-event failures.

In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed

BLUEPRINTS_DIR = "../../data/scenes/counselor_blueprints"
os.makedirs(BLUEPRINTS_DIR, exist_ok=True)

pending = [e for e in all_events
           if not os.path.exists(os.path.join(BLUEPRINTS_DIR, f"{e['id']}.json"))]
print(f"{len(all_events) - len(pending)} already saved, {len(pending)} remaining\n")

def create_counselor_blueprint_safe(event):
    try:
        return create_counselor_blueprint(event, all_events, autobiography)
    except Exception as e:
        print(f"  ERROR {event.get('id', '?')}: {e}")
        return None

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(create_counselor_blueprint_safe, e): e["id"] for e in pending}
    for future in as_completed(futures):
        event_id = futures[future]
        result = future.result()
        if result is None:
            print(f"  SKIPPED {event_id}")
            continue
        out_path = os.path.join(BLUEPRINTS_DIR, f"{event_id}.json")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        print(f"Saved {event_id}")

0 already saved, 39 remaining

Saved father_emotionally_absent_during_childhood
Saved parents_split_and_father_leaves_early_childhood
Saved father_hits_with_belt_age_3
Saved mother_emotionally_unavailable_in_childhood
Saved witnesses_father_beating_mother_in_three_room_house
Saved mother_works_late_and_brother_cares_for_her
Saved groomed_online_by_pedophiles_age_8
Saved blamed_for_grooming_after_age_8_abuse
Saved prays_not_to_exist_by_age_9
Saved early_suicidal_thought_in_frog_bathroom_age_5
Saved begins_self_harm_with_pain_methods_age_10
Saved father_abandons_family_and_steals_everything_age_9
Saved sexual_assault_not_believed_and_blamed_adolescence
Saved enters_secondary_school_and_faces_relentless_bullying
Saved sexual_assault_by_older_boy_age_12
Saved bullying_triggers_self_harm_in_school_years
Saved peer_takes_photos_and_laughs_at_her
Saved rumors_and_fake_chats_damage_reputation
Saved excluded_from_friend_groups_and_sits_alone_at_lunch
Saved adults_at_school_brush_off_bullying_re

## Merge + Validate

In [5]:
import glob

REQUIRED_KEYS = {"id", "event_id", "stages"}
REQUIRED_STAGES = {"beginning", "middle", "end"}

all_blueprints = []
for path in sorted(glob.glob(os.path.join(BLUEPRINTS_DIR, "*.json"))):
    with open(path, "r", encoding="utf-8") as f:
        bp = json.load(f)

    missing = REQUIRED_KEYS - bp.keys()
    if missing:
        print(f"WARNING: skipping {path} missing {missing}")
        continue

    missing_stages = REQUIRED_STAGES - bp["stages"].keys()
    empty_stages = {s for s in REQUIRED_STAGES - missing_stages if not bp["stages"][s].get("description")}
    if missing_stages or empty_stages:
        print(f"WARNING: skipping {path} missing_stages={missing_stages} empty_stages={empty_stages}")
        continue

    all_blueprints.append(bp)

OUT_PATH = "../../data/hannah_counselor_blueprints.json"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump({"blueprints": all_blueprints}, f, ensure_ascii=False, indent=2)

print(f"Saved {len(all_blueprints)} blueprints to {OUT_PATH}")

Saved 39 blueprints to ../../data/hannah_counselor_blueprints.json
